# The Larceny Wave: How Theft Shaped San Francisco's Streets (2003–2025)

## Introduction

San Francisco's Police Department (SFPD) records every reported incident in a publicly available dataset dating back to 2003. With over three million entries spanning more than two decades, this data captures the rhythms of urban crime: which neighborhoods see the most incidents, what time of day people are most at risk, and how crime patterns have shifted over the years.

This story focuses on **larceny theft** — by far the most common crime in the dataset, accounting for roughly 70% of the six focus crimes tracked in this analysis. Larceny covers a broad range of property crimes including pickpocketing, shoplifting, and the infamous car break-ins that became synonymous with San Francisco during the 2010s. Between 2003 and 2025, larceny theft incidents rose by more than 50%, peaked dramatically in the mid-2010s, and have only recently begun to fall back toward historical norms.

Three questions guide this data story:
1. **When did larceny theft surge — and why?**
2. **Where in the city does it concentrate?**
3. **When during the week is theft most likely?**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import folium
import json
from pathlib import Path

current_dir = Path.cwd()
data_path = current_dir.parent.parent / 'Data' / '2003_2026.csv'
shape_path = current_dir.parent.parent / 'Data' / 'sf_districts.geojson'

# Load data
data = pd.read_csv(data_path)
data['Incident Datetime'] = pd.to_datetime(data['Incident Datetime'])
data['Year'] = data['Incident Datetime'].dt.year
data['Hour'] = data['Incident Datetime'].dt.hour
data['DayOfWeek'] = data['Incident Datetime'].dt.day_name()

focus_crimes = ['Robbery', 'Motor Vehicle Theft', 'Weapons Offense',
                'Rape', 'Disorderly Conduct', 'Larceny Theft']
focus_data = data[data['Incident Category'].isin(focus_crimes)].copy()
larceny = focus_data[focus_data['Incident Category'] == 'Larceny Theft'].copy()

# Consistent styling
ACCENT_COLOR = '#C0392B'   # strong red for larceny bars
MUTED_COLOR  = '#BDC3C7'   # light grey for context bars
BG_COLOR     = '#FAFAFA'
FONT = 'DejaVu Sans'

plt.rcParams.update({
    'font.family': FONT,
    'axes.facecolor': BG_COLOR,
    'figure.facecolor': BG_COLOR,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.4,
    'grid.linestyle': '--',
})
print('Data loaded:', len(data), 'total incidents;', len(larceny), 'larceny incidents')

---
## Part 1 — Twenty Years of Theft

From 2003 through 2011, larceny theft in San Francisco was remarkably stable: roughly 24,000–27,000 incidents per year. Then something changed.

Starting in 2012, annual counts began climbing steadily — 30,760 in 2012, 36,216 in 2013, 41,980 in 2015 — culminating in a peak of **47,714 incidents in 2017**. This near-doubling over just five years was not an accident of data collection. It coincided directly with the peak of San Francisco's tech boom, a period of explosive economic growth that also deepened inequality and left hundreds of thousands of cars parked on city streets as workers commuted by rideshare or shuttle.

San Francisco's car break-in epidemic was widely covered in the press. In 2017 the city recorded more vehicle break-ins than any comparable US city, with tourists' rental cars on Fisherman's Wharf targeted so routinely that the San Francisco Chronicle ran headlines warning visitors to leave nothing in their vehicles. The SFPD logged **over 30,000 vehicle break-ins** in 2017 alone — incidents that fall under the larceny category in this dataset.

The COVID-19 pandemic imposed an abrupt brake. With lockdowns emptying streets, offices, and tourist areas in 2020, larceny theft plummeted to 30,618 — nearly back to the pre-boom baseline. The partial recovery in 2021–2023 suggests street activity resumed, but by 2024–2025 counts dropped again, likely reflecting both genuine declines and partially incomplete data for those years.

In [ ]:
yearly = larceny.groupby('Year').size().reset_index(name='Count')
# Exclude 2026 (incomplete)
yearly = yearly[yearly['Year'] <= 2025]

fig, ax = plt.subplots(figsize=(12, 5))

# Color bars: highlight surge years 2012-2019 in accent, rest muted
colors = [ACCENT_COLOR if 2012 <= y <= 2019 else MUTED_COLOR for y in yearly['Year']]

bars = ax.bar(yearly['Year'], yearly['Count'], color=colors, width=0.7, zorder=3)

# Annotations
ax.annotate(
    'Surge begins\n(tech boom era)',
    xy=(2012, 30760), xytext=(2009.5, 38000),
    arrowprops=dict(arrowstyle='->', color='#555', lw=1.2),
    fontsize=9, color='#333'
)
ax.annotate(
    'Peak: 47,714',
    xy=(2017, 47714), xytext=(2017, 51000),
    ha='center', fontsize=9, color=ACCENT_COLOR, fontweight='bold'
)
ax.annotate(
    'COVID-19\nlockdowns',
    xy=(2020, 30618), xytext=(2021.5, 24500),
    arrowprops=dict(arrowstyle='->', color='#555', lw=1.2),
    fontsize=9, color='#333'
)

# Legend: pass patches as handles, do NOT add_patch() them
legend_handles = [
    mpatches.Patch(color=ACCENT_COLOR, label='Surge years (2012\u20132019)'),
    mpatches.Patch(color=MUTED_COLOR, label='Other years'),
]
ax.legend(handles=legend_handles, frameon=False, fontsize=9, loc='upper left')

ax.set_xlabel('Year', fontsize=11)
ax.set_ylabel('Larceny Theft Incidents', fontsize=11)
ax.set_title('Larceny Theft in San Francisco, 2003\u20132025', fontsize=13, fontweight='bold', pad=12)
ax.set_xlim(2002.3, 2025.7)
ax.set_xticks(yearly['Year'])
ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('larceny_timeseries.png', dpi=150, bbox_inches='tight')
plt.show()

**Figure 1.** Annual larceny theft incidents in San Francisco, 2003–2025. Bars highlighted in red mark the surge period (2012–2019). The city's tech-boom years are clearly reflected in the data: counts nearly doubled between 2011 and 2017. The steep drop in 2020 aligns exactly with COVID-19 lockdowns, while 2024–2025 figures may reflect partially incomplete data for those years.

---
## Part 2 — Where Theft Concentrates

Larceny theft is not distributed evenly across San Francisco's eleven police districts. The map below shows each district's **relative risk ratio** — how many larceny thefts occurred per 10,000 residents relative to the city-wide average. A ratio above 1.0 means that district experiences more larceny than average; below 1.0 means less.

The results are striking. The **Central district** — encompassing the Financial District, Union Square, and Chinatown — tops the list with a risk ratio nearly three times the city average. This is not surprising: these are densely packed commercial and tourist areas where crowds, opportunity, and high-value targets converge. Fisherman's Wharf and Market Street, two of SF's most visited corridors, both fall within or adjacent to Central.

The **Northern district** (covering the Marina, Russian Hill, and Civic Center) ranks second, also driven by dense foot traffic and concentrations of parked vehicles. By contrast, residential districts like **Taraval** (West Portal, Sunset) and **Park** (Inner Sunset, Haight) see larceny rates closer to or below the city average — consistent with quieter residential streets and less tourist activity.

One notable outlier: the **Tenderloin**, despite being a dense, high-foot-traffic neighborhood with well-documented social challenges, has a larceny rate *below* average. This is partly because larceny theft is opportunity-driven — it peaks where tourists and commuters leave valuables in cars or bags. The Tenderloin's residents are far less likely to be carrying laptops or leaving rental cars stocked with luggage.

In [ ]:
# Load district GeoJSON
with open(shape_path, 'r', encoding='utf-8') as f:
    districts_geojson = json.load(f)

# Get district population estimates (approximate 2020 Census figures per district)
district_population = {
    'CENTRAL': 72000, 'SOUTHERN': 105000, 'BAYVIEW': 68000,
    'MISSION': 80000, 'NORTHERN': 90000, 'INGLESIDE': 70000,
    'TARAVAL': 80000, 'RICHMOND': 78000, 'TENDERLOIN': 35000,
    'PARK': 60000
}

# Count larceny by district (use data from all years)
larceny_valid = larceny[larceny['Police District'].notna()].copy()
larceny_valid['District_upper'] = larceny_valid['Police District'].str.upper()
district_counts = larceny_valid.groupby('District_upper').size().reset_index(name='larceny_count')

# City-wide rate per 10k residents  
city_total_pop = sum(district_population.values())
city_total_larceny = district_counts['larceny_count'].sum()
city_rate = city_total_larceny / city_total_pop * 10000

# Compute risk ratio
district_counts['population'] = district_counts['District_upper'].map(district_population)
district_counts = district_counts.dropna(subset=['population'])
district_counts['rate_per_10k'] = district_counts['larceny_count'] / district_counts['population'] * 10000
district_counts['risk_ratio'] = district_counts['rate_per_10k'] / city_rate

ratio_dict = dict(zip(district_counts['District_upper'], district_counts['risk_ratio']))

print('Larceny risk ratios by district:')
for d, r in sorted(ratio_dict.items(), key=lambda x: -x[1]):
    print(f'  {d:<15} {r:.2f}x city average')

In [ ]:
# Build Folium choropleth map
m = folium.Map(location=[37.7749, -122.4194], zoom_start=12,
               tiles='CartoDB positron')

# Inject risk ratio into GeoJSON properties for choropleth
for feature in districts_geojson['features']:
    dist_name = feature['properties']['DISTRICT']
    feature['properties']['risk_ratio'] = ratio_dict.get(dist_name, 0)

folium.Choropleth(
    geo_data=districts_geojson,
    data=district_counts,
    columns=['District_upper', 'risk_ratio'],
    key_on='feature.properties.DISTRICT',
    fill_color='YlOrRd',
    fill_opacity=0.75,
    line_opacity=0.5,
    legend_name='Larceny Theft Risk Ratio (vs city average)',
    nan_fill_color='lightgrey'
).add_to(m)

# Add district name labels
for feature in districts_geojson['features']:
    coords = feature['geometry']['coordinates']
    # Flatten for polygon/multipolygon
    if feature['geometry']['type'] == 'Polygon':
        pts = coords[0]
    else:
        pts = coords[0][0]
    lats = [p[1] for p in pts]
    lons = [p[0] for p in pts]
    center = [np.mean(lats), np.mean(lons)]
    dist = feature['properties']['DISTRICT']
    ratio = ratio_dict.get(dist, 0)
    folium.Marker(
        location=center,
        icon=folium.DivIcon(
            html=f'<div style="font-size:10px;font-weight:bold;color:#222;"\
                  text-shadow:0 0 3px #fff;">{dist}<br>{ratio:.1f}x</div>',
            icon_size=(80, 30),
            icon_anchor=(40, 15)
        )
    ).add_to(m)

m.save('larceny_map.html')
m

**Figure 2.** Larceny theft risk ratio by SF police district, 2003–2025 (darker = higher risk). Each district's label shows its risk ratio relative to the city average (1.0 = average). The Central district — home to Union Square, the Financial District, and major tourist corridors — sees larceny at nearly three times the city average. Population estimates used are approximate 2020 Census figures.

---
## Part 3 — The Thief's Clock: When Theft Happens

Knowing *where* theft concentrates tells only half the story. The interactive heatmap below reveals *when* — broken down by both hour of day and day of week across the full 2003–2025 dataset.

The pattern is unmistakable: larceny theft follows the rhythms of urban life with precision. Counts are lowest in the early morning hours (2–6 AM), when streets are quiet and most potential victims are asleep. Activity begins climbing as the city wakes, peaks sharply in the **afternoon window between noon and 6 PM**, and tapers off into the evening.

This afternoon peak reflects the confluence of opportunity factors: lunch crowds spilling onto sidewalks, commuters rushing between transit stops, tourists unloading luggage from parked rental cars. The SFPD has long noted that smash-and-grab thefts peak in daylight hours rather than at night — a counter-intuitive finding for many residents who assume darkness equals danger.

**Friday and Saturday afternoons** stand out as the highest-risk window in the entire week. The combination of weekend leisure, tourist concentration, and evening social activity creates a perfect storm for opportunistic theft. By contrast, Tuesday and Wednesday mornings are among the safest times in the city — a finding consistent with routine activity theory, which predicts crime rates track the overlap between motivated offenders and available targets.

In [ ]:
from bokeh.plotting import figure, output_notebook, show
from bokeh.models import (
    ColumnDataSource, LinearColorMapper, ColorBar,
    BasicTicker, HoverTool
)
from bokeh.palettes import YlOrRd9
from bokeh.io import export_png
import warnings
warnings.filterwarnings('ignore')

output_notebook()

# Build day × hour counts
days_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
hours_order = [str(h) for h in range(24)]

hm = (
    larceny
    .groupby(['DayOfWeek', 'Hour'])
    .size()
    .reset_index(name='Count')
)
hm = hm[hm['DayOfWeek'].isin(days_order)]
hm['HourStr'] = hm['Hour'].astype(str)

# Color mapper
mapper = LinearColorMapper(
    palette=list(reversed(YlOrRd9)),
    low=hm['Count'].min(),
    high=hm['Count'].max()
)

source = ColumnDataSource(hm)

p = figure(
    title='Larceny Theft by Hour and Day of Week (2003–2025)',
    x_range=hours_order,
    y_range=list(reversed(days_order)),
    width=820, height=320,
    toolbar_location='above',
    x_axis_label='Hour of Day',
    y_axis_label='Day of Week'
)

p.rect(
    x='HourStr', y='DayOfWeek',
    width=1, height=1,
    source=source,
    fill_color={'field': 'Count', 'transform': mapper},
    line_color=None
)

# Hover tool
hover = HoverTool(tooltips=[
    ('Day',   '@DayOfWeek'),
    ('Hour',  '@Hour{00}:00'),
    ('Total incidents', '@Count{,}')
])
p.add_tools(hover)

# Color bar
color_bar = ColorBar(
    color_mapper=mapper,
    ticker=BasicTicker(desired_num_ticks=6),
    label_standoff=8,
    border_line_color=None,
    location=(0, 0),
    title='Incidents'
)
p.add_layout(color_bar, 'right')

# Style
p.axis.axis_line_color = None
p.axis.major_tick_line_color = None
p.axis.major_label_text_font_size = '10px'
p.axis.major_label_standoff = 0
p.title.text_font_size = '13px'
p.title.text_font_style = 'bold'
p.background_fill_color = BG_COLOR
p.border_fill_color = BG_COLOR
p.grid.grid_line_color = None

show(p)

In [ ]:

# ── Export real data for pages/larceny-story.html ──────────────────────────
# Run this cell after the three visualisation cells above.
# It writes larceny_data.js to the repo root; the HTML page picks it up
# automatically and replaces the placeholder data with real values.
import json

# Figure 1: yearly counts (already computed above as `yearly`)
years_list  = [int(y)  for y in yearly['Year'].tolist()]
counts_list = [int(c)  for c in yearly['Count'].tolist()]

# Figure 2: district risk ratios (sorted high → low)
districts_sorted = district_counts.sort_values('risk_ratio', ascending=False)
districts_list = districts_sorted['District_upper'].tolist()
ratios_list    = [round(float(r), 3) for r in districts_sorted['risk_ratio'].tolist()]

# Figure 3: day × hour heatmap (already computed above as `hm`)
days_order_export = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
heatmap_list = []
for day in days_order_export:
    row = []
    for h in range(24):
        sub = hm[(hm['DayOfWeek'] == day) & (hm['Hour'] == h)]
        row.append(int(sub['Count'].values[0]) if len(sub) > 0 else 0)
    heatmap_list.append(row)

js_content = (
    "// Auto-generated by things_to_put_on_website.ipynb — do not edit by hand.\n"
    "// Re-run the notebook to regenerate with fresh data.\n"
    "const LARCENY_DATA = {\n"
    f"  years:     {json.dumps(years_list)},\n"
    f"  counts:    {json.dumps(counts_list)},\n"
    f"  districts: {json.dumps(districts_list)},\n"
    f"  ratios:    {json.dumps(ratios_list)},\n"
    f"  heatmap:   {json.dumps(heatmap_list)}\n"
    "};\n"
)

output_path = current_dir / 'larceny_data.js'
with open(output_path, 'w') as f:
    f.write(js_content)

print(f"Saved: {output_path}")
print(f"  Figure 1 — {len(years_list)} years ({years_list[0]}–{years_list[-1]})")
print(f"  Figure 2 — {len(districts_list)} districts")
print(f"  Figure 3 — heatmap {len(heatmap_list)} days × 24 hours")
print("\nThe page will use real data automatically once larceny_data.js is in the repo root.")


**Figure 3** *(interactive)*. Larceny theft heatmap by hour of day (x-axis) and day of week (y-axis), 2003–2025. Darker red cells indicate more incidents. Hover over any cell to see exact counts. The afternoon surge (12:00–18:00) is visible across all days, with Friday and Saturday afternoons representing the city's highest-risk windows. Early morning hours (2–6 AM) are consistently the safest.

---
## Conclusion

Larceny theft in San Francisco is not random. It concentrates in time (weekday and weekend afternoons), in space (commercial and tourist-heavy districts), and in history (the tech-boom years of 2012–2018 saw a dramatic surge that has since partially receded).

The data tells a coherent story: as the city's economy boomed, so did the density of targets — tourists, commuters, and well-stocked parked cars. When COVID-19 emptied those same streets in 2020, theft collapsed alongside foot traffic. The partial recovery in 2021–2023, and the renewed decline in 2024–2025, mirror broader trends in San Francisco's post-pandemic urban landscape.

What the data cannot tell us is why the city failed to translate economic prosperity into public safety during the surge years, or whether the recent declines reflect genuine policing success, demographic shifts, or simply fewer people carrying valuables on city streets. For those answers, we would need to look beyond crime statistics into the city's housing, economic, and social policy records — a story for another dataset, another day.

---
### References
1. San Francisco Police Department, *SFPD Incident Reports: 2003 to Present*, DataSF Open Data Portal.  
2. Levin, S. (2017, October 16). ["San Francisco has more car break-ins than any other major US city."](https://www.theguardian.com/us-news/2017/oct/16/san-francisco-car-break-ins-new-high) *The Guardian*.  
3. Ferré-Sadurní, L. (2018, January 20). ["San Francisco's boom-era safety dilemma."](https://www.sfchronicle.com) *San Francisco Chronicle*.  
4. Cohen, L. E., & Felson, M. (1979). Social change and crime rate trends: A routine activity approach. *American Sociological Review*, 44(4), 588–608.